# 🥬 Estimasi Waktu Panen Selada Hidroponik Menggunakan Machine Learning
**Project-Based Learning (PBL) — Kelompok 5**
*Mata Kuliah: Pembelajaran Mesin, Pemrograman Mobile, Penjaminan Mutu Perangkat Lunak*

Notebook ini mencakup tiga pilar Machine Learning:
1. **Regresi**: Estimasi hari menuju panen (`days_left`)
2. **Klasifikasi**: Prediksi fase pertumbuhan dan kesiapan panen (*Semai, Vegetatif, Siap Panen*)
3. **Klusterisasi**: Segmentasi profil mikroklimat lingkungan tanaman (*K-Means*)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
import joblib

print('Semua library berhasil diimpor!')

## 1. Load Dataset & Preprocessing
Dataset pertumbuhan selada hidroponik dimuat dari `lettuce_dataset_training.csv`.

In [ ]:
col_names = ['Plant_ID', 'Date', 'Temperature_C', 'Humidity_pct', 'TDS_ppm', 'pH', 'Growth_Days', 'Temperature_F', 'Humidity_ratio']
csv_path = 'data/lettuce_dataset_training.csv'
if not os.path.exists(csv_path):
    csv_path = 'ML/data/lettuce_dataset_training.csv'

raw = pd.read_csv(csv_path, header=None, names=col_names)
df = pd.DataFrame({
    'plant_id': raw['Plant_ID'],
    'umur_hst': raw['Growth_Days'].astype(float),
    'suhu': raw['Temperature_C'].astype(float),
    'kelembapan': raw['Humidity_pct'].astype(float),
    'tds': raw['TDS_ppm'].astype(float),
    'ph': raw['pH'].astype(float)
})

# Hitung target regresi days_left
harvest_days = df.groupby('plant_id')['umur_hst'].transform('max')
df['days_left'] = np.clip(harvest_days - df['umur_hst'], 0, None)

# Hitung target klasifikasi fase: 0: Semai, 1: Vegetatif, 2: Siap Panen
conditions = [
    (df['umur_hst'] <= 14),
    (df['umur_hst'] > 14) & (df['umur_hst'] <= 35),
    (df['umur_hst'] > 35)
]
df['stage_class'] = np.select(conditions, [0, 1, 2], default=1)
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print(df.info())
df.describe()

## 3. Pelatihan & Evaluasi Model Regresi (Estimasi Sisa Hari Panen)

In [ ]:
features = ['umur_hst', 'suhu', 'kelembapan', 'tds', 'ph']
X = df[features]
y_reg = df['days_left']
y_cls = df['stage_class']

X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

reg_models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

for name, m in reg_models.items():
    m.fit(X_train, yr_train)
    preds = m.predict(X_test)
    mae = mean_absolute_error(yr_test, preds)
    rmse = np.sqrt(mean_squared_error(yr_test, preds))
    r2 = r2_score(yr_test, preds)
    print(f'{name:22} MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f}')

## 4. Pelatihan & Evaluasi Model Klasifikasi (Fase Pertumbuhan / Kesiapan Panen)
Membandingkan: KNN, SVM, Naive Bayes, dan Random Forest Classifier.

In [ ]:
cls_models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(probability=True, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, m in cls_models.items():
    m.fit(X_train, yc_train)
    preds = m.predict(X_test)
    acc = accuracy_score(yc_test, preds)
    prec = precision_score(yc_test, preds, average='macro', zero_division=0)
    rec = recall_score(yc_test, preds, average='macro', zero_division=0)
    f1 = f1_score(yc_test, preds, average='macro', zero_division=0)
    print(f'{name:15} Akurasi: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')

## 5. Pemodelan Klusterisasi Lingkungan Tanaman (K-Means)
Segmentasi kondisi nutrisi dan iklim selada (suhu, kelembapan, TDS, pH) ke dalam 3 cluster profil.

In [ ]:
env_features = ['suhu', 'kelembapan', 'tds', 'ph']
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(df[env_features])
sil = silhouette_score(df[env_features], labels)
print(f'Silhouette Score K-Means (k=3): {sil:.4f}')
for c in range(3):
    sub = df[labels == c][env_features]
    print(f'Cluster {c} (n={len(sub)}): Suhu={sub["suhu"].mean():.1f}C, Kelembapan={sub["kelembapan"].mean():.1f}%, TDS={sub["tds"].mean():.1f}, pH={sub["ph"].mean():.2f}')